# 243 - archetypal analysis

**A fourth track, next to k-means, Ward and convex NMF.** Not a variant of them: the one
thing it changes is where the components are allowed to sit.

| | convex NMF | archetypal analysis |
|---|---|---|
| model | `X ~= G (W'X)` | `X ~= A (B X)` |
| components | `W >= 0` — a weighted average of electrodes | `B >= 0` **and each row sums to 1** |
| memberships | `G >= 0`, renormalised afterwards to read as proportions | `A >= 0` **and each row sums to 1**, so they ARE proportions as fitted |
| where a component lands | the **interior** of the cloud | the **convex hull** of it |

An average of a mixed population looks like the population, so a convex-NMF component
tends towards a watered-down version of the typical response. An archetype is an
**extreme** - the purest auditory response present in the cohort - and every electrode is
then written as a mixture of extremes whose weights sum to one.

**What that buys.** An electrode's loading vector reads directly as a sentence about the
electrode: *60% auditory archetype, 40% reading archetype*. Convex NMF's `G` has to be
divided by its row sum before it means that, and that division is a step where the
meaning can quietly change.

**What it risks, and how this notebook checks it.** Archetypes sit at the edge, so an
archetype can be one outlier wearing a hat. `archetype_support()` reports the *effective
support* of each - the exponential of the entropy of its weights - which is literally
"how many electrodes is this archetype actually averaging". A value near 1 means a single
electrode IS the archetype. **This is the check that decides whether a K is usable**, and
none of the other three methods has an equivalent.

**Before this:** 240, 241, 242 and the cache rebuild.
**Method:** Cutler & Breiman (1994), projected gradient with exact simplex projection
(Duchi et al. 2008), FurthestSum initialisation (Morup & Hansen 2012).


In [ ]:
import os, sys, json, subprocess, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
assert (ROOT / 'functions').exists(), f'run this from 02_FBM_Clustering, not {ROOT}'
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / 'functions'))
import lf_archetypes as AA, lf_decompose as LD

CLUST = ROOT / 'outputs' / 'clustering'
FEATURE_SETS = ['concat_hg', 'concat_rawds', 'concat_bands5']

# the K each feature set is cut at, from the convex-NMF held-out peak that 249 picked
PEAK_K = json.loads((CLUST / 'bsf_comparison' / 'peak_k.json').read_text())
print('cut at:', PEAK_K)

def sh(args):
    print('$', ' '.join(str(a) for a in args)); t = time.time()
    r = subprocess.run([sys.executable, *[str(a) for a in args]], cwd=ROOT,
                       env={**os.environ, 'PYTHONIOENCODING': 'utf-8'})
    print(f'  exit {r.returncode}  ({time.time()-t:.0f}s)')
    assert r.returncode == 0, args
    return r

def newest(method, fset):
    d = CLUST / method / fset / 'runs'
    return sorted(p for p in d.iterdir() if p.is_dir())[-1] if d.is_dir() else None

def unit(A):
    return A / np.maximum(np.linalg.norm(A, axis=1, keepdims=True), 1e-12)


## 1 - Fit

`run_archetypes.py` copies `X_train.npy` byte-for-byte from the k-means run each
decomposition is derived from, so the archetype run and the convex-NMF run it is compared
against are guaranteed to hold **the same electrodes in the same order**. That is asserted
below rather than assumed.

~25 minutes per feature set for K = 5..30.


In [ ]:
sh(['run_archetypes.py'])          # both feature sets, K = 5..30

for fs in FEATURE_SETS:
    a, c = newest('archetypes', fs), newest('cnmf', fs)
    Xa, Xc = np.load(a / 'X_train.npy'), np.load(c / 'X_train.npy')
    assert np.array_equal(Xa, Xc), f'{fs}: archetypes and cnmf are NOT the same cohort'
    print(f'  {fs:<14} {Xa.shape[0]} electrodes, identical to {c.name}')


## 2 - The sweep, and where archetypes stop being real

Two curves, and the second is the one that matters.

**Variance explained** rises with K, as it must - more components always fit better.

**Minimum effective support** falls with K, and that is the stopping rule. Once the
smallest archetype rests on a handful of electrodes it is describing those electrodes and
not a response type. There is no equivalent number for k-means or Ward: a centroid is
always the mean of its whole cluster, so it cannot degenerate this way, which is exactly
why the degeneracy has to be checked here and not there.


In [ ]:
fig, axes = plt.subplots(2, len(FEATURE_SETS), figsize=(6.4 * len(FEATURE_SETS), 7.4),
                         sharex=True)
axes = np.atleast_2d(axes)
SWEEPS = {}
for j, fs in enumerate(FEATURE_SETS):
    rd = newest('archetypes', fs)
    s = pd.read_csv(rd / 'sweep_by_k.csv').sort_values('k')
    SWEEPS[fs] = s
    kcut = PEAK_K.get(fs)

    ax = axes[0, j]
    ax.plot(s.k, s.var_explained, '-o', ms=3.4, color='#5b2c83')
    if not s.converged.all():
        bad = s[~s.converged]
        ax.plot(bad.k, bad.var_explained, 'x', ms=9, color='#c1121f',
                label='hit the iteration cap')
        ax.legend(fontsize=7.5, frameon=False)
    ax.axvline(kcut, color='#c1121f', ls='--', lw=1.0)
    ax.set_title(f'{fs}  ·  variance explained', fontsize=10.5, loc='left')
    ax.set_ylabel('var explained'); ax.spines[['top','right']].set_visible(False)

    ax = axes[1, j]
    ax.plot(s.k, s.min_effective_support, '-o', ms=3.4, color='#1b7837',
            label='smallest archetype')
    ax.plot(s.k, s.median_effective_support, '-o', ms=2.6, color='#68727d', lw=1.0,
            label='median')
    ax.axhline(3, color='#c1121f', ls=':', lw=1.2)
    ax.text(s.k.max(), 3.1, 'below this an archetype is a few electrodes',
            fontsize=7.4, color='#c1121f', ha='right')
    ax.axvline(kcut, color='#c1121f', ls='--', lw=1.0)
    ax.set_yscale('log')
    ax.set_title(f'{fs}  ·  effective support (electrodes per archetype)',
                 fontsize=10.5, loc='left')
    ax.set_xlabel('K'); ax.set_ylabel('electrodes'); ax.legend(fontsize=7.5, frameon=False)
    ax.spines[['top','right']].set_visible(False)

    ok = s[s.min_effective_support >= 3]
    kmax = int(ok.k.max()) if len(ok) else None
    print(f'{fs:<14} cut at K={kcut}: var {float(s.loc[s.k==kcut,"var_explained"].iloc[0]):.4f}, '
          f'smallest archetype {float(s.loc[s.k==kcut,"min_effective_support"].iloc[0]):.1f} electrodes')
    print(f'{"":<14} last K where every archetype has >=3 electrodes: {kmax}')
plt.tight_layout()


## 3 - The claim, measured

Archetypes are supposed to sit **further out** than centroids and than convex-NMF
components. If they do not, the whole exercise bought nothing and this notebook should
say so.

The comparison is **archetypes against convex-NMF components only**, and that is
deliberate. k-means centroids live in raw dB; unit-norming them to put them on this axis
projects them onto the sphere, where their distance from the centre is set by the
projection rather than by where they sit. Measured that way they come out at 0.861 -
further out than the archetypes - which is an artefact of the normalisation and not a
fact about k-means. Two methods can only be compared on this axis if they were both
fitted on it.


In [ ]:
rows = []
for fs in FEATURE_SETS:
    K = PEAK_K[fs]
    ra, rc = newest('archetypes', fs), newest('cnmf', fs)
    Xu = unit(np.load(ra / 'X_train.npy').astype(float))
    ctr = Xu.mean(0)

    Z = np.load(ra / 'components_by_k' / f'C_k{K:02d}.npy').astype(float)
    G = np.load(rc / 'loadings_by_k' / f'G_k{K:02d}.npy').astype(float)
    # the cNMF component profiles at this K, rebuilt the way the run defines them
    Cc = np.load(rc / 'components_by_k' / f'C_k{K:02d}.npy').astype(float)

    d = lambda M: float(np.linalg.norm(np.asarray(M) - ctr, axis=1).mean())
    rows.append(dict(feature_set=fs, K=K,
                     archetypes=d(Z), cnmf_components=d(Cc), electrodes=d(Xu),
                     electrode_max=float(np.linalg.norm(Xu - ctr, axis=1).max())))

hull = pd.DataFrame(rows)
print(hull.to_string(index=False))
print()
for _, r in hull.iterrows():
    ok = r.archetypes > r.cnmf_components
    print(f'  {r.feature_set}: archetypes {"ARE" if ok else "are NOT"} further out than '
          f'the convex-NMF components — {r.archetypes:.3f} vs {r.cnmf_components:.3f}, '
          f'against {r.electrodes:.3f} for a typical electrode and {r.electrode_max:.3f} '
          f'for the furthest one.')
    print(f'  {"":>{len(r.feature_set)}}  so the archetypes reach '
          f'{100*r.archetypes/r.electrodes:.0f}% of a typical electrode's distance from '
          f'the centre and the components reach {100*r.cnmf_components/r.electrodes:.0f}%.')


## 4 - Does it find anything the others did not?

Three questions, the same ones the other methods are held to in 249.

1. **Agreement** - if archetypes reproduce the convex-NMF partition, the constraint
   changed nothing worth having.
2. **Gradedness** - what fraction of electrodes have no majority archetype. A *high*
   number is expected here and is not a fault: if every electrode sat at one extreme
   there would be no mixing to model.
3. **Anatomy** - neighbours sharing a label over chance, the one criterion that is about
   the brain rather than about the fit.


In [ ]:
from sklearn.metrics import adjusted_rand_score as ari, normalized_mutual_info_score as nmi

out = []
for fs in FEATURE_SETS:
    K = PEAK_K[fs]
    ra, rc = newest('archetypes', fs), newest('cnmf', fs)
    A = np.load(ra / 'loadings_by_k' / f'A_k{K:02d}.npy').astype(float)
    G = np.load(rc / 'loadings_by_k' / f'G_k{K:02d}.npy').astype(float)
    la = A.argmax(1)
    lc = (G / np.maximum(G.sum(1, keepdims=True), 1e-12)).argmax(1)
    lk = pd.read_csv(newest('kmeans', fs) / 'cluster_labels_by_k.csv')[f'k_{K}'].to_numpy()

    # COORDINATES COME FROM THE STATISTICS DIR, not from labels.csv - labels.csv
    # carries no x/y/z at all. 249 already joined them onto this exact cohort in this
    # exact row order, and the archetype run shares that X_train byte-for-byte, so the
    # array lines up; the assert is what makes that a fact rather than a hope.
    sx = CLUST / 'statistics' / f'{fs}_K{K}' / 'stats_xyz.npy'
    coh_a = coh_c = np.nan
    if sx.exists():
        xyz = np.load(sx)
        assert len(xyz) == len(la), f'{fs}: xyz {len(xyz)} vs labels {len(la)}'
        m = np.isfinite(xyz).all(1)
        # spatial_coherence returns (observed, over_chance) - the ratio is the one to
        # quote, because the raw number rewards one dominant cluster for free
        coh_a = LD.spatial_coherence(la[m], xyz[m])[1]
        coh_c = LD.spatial_coherence(lc[m], xyz[m])[1]
    else:
        print(f'  {fs}: no stats_xyz.npy - run 249 first for the coherence column')

    out.append(dict(feature_set=fs, K=K,
                    ari_vs_cnmf=ari(la, lc), nmi_vs_cnmf=nmi(la, lc),
                    ari_vs_kmeans=ari(la, lk),
                    frac_no_majority_arch=float((A.max(1) < 0.5).mean()),
                    frac_no_majority_cnmf=float(
                        ((G / np.maximum(G.sum(1, keepdims=True), 1e-12)).max(1) < 0.5).mean()),
                    coherence_arch=coh_a, coherence_cnmf=coh_c))

cmp = pd.DataFrame(out)
pd.set_option('display.width', 170)
print(cmp.to_string(index=False))


## 5 - Read this before quoting any of it

- **Archetypes are not cluster centres**, so reporting a run as a partition by taking
  the argmax throws away the thing it was fitted to produce - the same argmax-vs-threshold
  trap convex NMF has.
- **But the memberships came out MORE decisive, not less.** The expectation before
  measuring was that few electrodes would sit near an extreme and `top_weight` would be
  low. The opposite is the case: on `concat_hg` at K=11, **40% of electrodes have no
  majority archetype against 91% for convex NMF**. Writing an electrode as a mixture of
  extremes turns out to be an easier description than writing it as a mixture of averages,
  and the section above measures that rather than assuming it either way.
- **`min_effective_support` is the gate.** Any K where the smallest archetype rests on
  fewer than ~3 electrodes is describing individuals, and the sweep above shows exactly
  where that starts.
- **Variance explained is not comparable to convex NMF's at face value.** Archetypal
  analysis converges here in a few hundred iterations while convex NMF at this project's
  `n_iter` has not converged, so a straight comparison of the two numbers measures the
  optimisers as much as the models. Compare them at matched convergence or not at all.
